# Part 11 — Long-Context LLM vs RAG, head-to-head

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mftnakrsu/rag-by-hand/blob/main/part-11-evaluating-rag/long_context_vs_rag.ipynb)

*"Context windows are huge now, so RAG is dead." The honest, runnable answer: it depends.*

📖 Read the essay: https://www.mefby.com/essays/evaluating-rag

🐍 Companion script: `long_context_vs_rag.py`

**What you'll build**

- A tiny **fictional** corpus ("Starlight Academy") with inserted **needles** — facts written in invented tokens so no pretrained knowledge can leak the answer.
- `llm_alone(question, corpus)` — stuff the **whole** corpus into the context window; cost = every token, every query.
- `rag(question, corpus, k)` — retrieve the top-k chunks by a deterministic embedder's cosine; cost = retrieved tokens only.
- A score table — **accuracy / cost(tokens) / latency(proxy)** — across two corpus sizes, so you can watch the tradeoff move as the haystack grows.

> **Runs fully offline.** Pure Python standard library + numpy. No API key, no model download, no network. There is no real LLM here on purpose: both strategies "answer" by the same transparent substring oracle, which isolates the one variable we care about — *what text each strategy got to see* — from the messiness of a real generator.

This is a companion experiment to Part 11. The debate it settles is **not** "RAG is dead" and **not** "RAG always wins" — it is the least satisfying answer in engineering: **it depends**, on corpus size, query type, and budget.

## The trap: data leakage

If you ask a real model "what year was the French Revolution", it answers from **pretraining**, not from your documents — and then "long context" looks magically accurate for the wrong reason. The fix (used by the **U-NIAH** benchmark, arXiv 2503.00353) is a **synthetic, fictional** corpus with inserted **needles**: facts stated in unique invented tokens that appear in no training set anywhere. If the answer is right, it came from the text in front of the model — never from memorized knowledge.

So we build a tiny fictional world. Each needle's question reuses the needle's rare invented tokens, so retrieval has a fair shot at finding it.

In [ ]:
import re
import numpy as np

# Each needle fact uses invented tokens (Zephyrine, Quorvax, glimwort, ...) that
# no pretrained model has seen -> a correct answer can ONLY come from the text.
NEEDLES = [
    {
        "question": "What is the name of the headmaster of Starlight Academy?",
        "fact": "The headmaster of Starlight Academy is Professor Zephyrine Quorvax.",
        "needle": "Zephyrine Quorvax",
    },
    {
        "question": "Which potion does the Glimwort Ceremony require?",
        "fact": "The Glimwort Ceremony at Starlight Academy requires the Moonsilver Draught.",
        "needle": "Moonsilver Draught",
    },
    {
        "question": "What is the secret password to the Astral Library?",
        "fact": "The secret password to the Astral Library is 'quorvex-lumens-7'.",
        "needle": "quorvex-lumens-7",
    },
]

for ni, nd in enumerate(NEEDLES):
    print(f"Q{ni + 1}: {nd['question']}")
    print(f"     gold needle -> {nd['needle']!r}")

## The haystack

To simulate a bigger corpus, we surround each needle with bland **filler** sentences. They share the world's vocabulary but answer nothing — they are the haystack. `build_corpus(n)` drops in every needle fact, then pads each with `n` rotating filler chunks. Turning `n` up is how we grow the corpus without ever adding a second answer.

In [ ]:
FILLER = [
    "Starlight Academy sits on a floating island above the Cinderhaven Sea.",
    "Students at the academy wear robes dyed with crushed glimwort petals.",
    "The east tower houses the observatory and three brass orreries.",
    "Every autumn the academy hosts a lantern regatta on the inner lake.",
    "First-year students are sorted into one of the four star-houses.",
    "The dining hall serves spiced cloudberry tart on festival evenings.",
    "A statue of the academy's founder stands in the central courtyard.",
    "The greenhouse cultivates rare moonsilver vines along its north wall.",
    "Owls deliver the academy post twice daily, at dawn and at dusk.",
    "The bell in the west tower has rung on the hour for four centuries.",
    "Apprentice alchemists practice in the vaulted basement laboratories.",
    "The academy crest shows a comet crossing a crescent moon.",
]


def build_corpus(n_filler_per_needle):
    """Each needle fact, padded with filler so the needle is buried in a haystack
    of size we control. Exactly len(NEEDLES) chunks carry an answer."""
    corpus, cid = [], 0
    for ni, needle in enumerate(NEEDLES):
        corpus.append({"id": f"c{cid}", "text": needle["fact"], "needle_for": ni})
        cid += 1
        for j in range(n_filler_per_needle):
            corpus.append({"id": f"c{cid}", "text": FILLER[(ni + j) % len(FILLER)],
                           "needle_for": None})
            cid += 1
    return corpus


small = build_corpus(n_filler_per_needle=3)
print(f"{len(small)} chunks; {sum(1 for c in small if c['needle_for'] is not None)} carry a needle")
for c in small[:5]:
    print(" ", c["id"], "->", c["text"][:60])

## Tokenizer + a deterministic offline embedder

We need two cheap, transparent helpers. A **tokenizer** that splits text into word-ish tokens (keeping hyphenated codes like `quorvex-lumens-7` intact), and an **embedder** — the same hashing stand-in the rest of the series uses (Part 2): any text becomes a fixed-length unit vector, reproducible with no model and no network. Texts that share words land closer, which is enough to retrieve a needle whose question reuses its rare tokens.

In [ ]:
DIM = 256


def tokenize(text):
    # keep hyphenated codes like 'quorvex-lumens-7' intact
    return re.findall(r"[a-z0-9]+(?:-[a-z0-9]+)*", text.lower())


def count_tokens(text):
    return len(tokenize(text))


def embed(text):
    """Deterministic hashing embedder: sum a stable pseudo-random direction per
    token, then L2-normalize. No model, no network -- same shape as Part 2."""
    vec = np.zeros(DIM)
    for tok in tokenize(text):
        seed = int.from_bytes(tok.encode("utf-8"), "little") % (2**32)
        vec += np.random.default_rng(seed).standard_normal(DIM)
    norm = np.linalg.norm(vec)
    return vec if norm == 0 else vec / norm


def cosine(a, b):
    return float(np.dot(a, b))   # unit vectors -> dot == cosine


q = NEEDLES[2]["question"]
print("query:", q)
print("cosine to the matching fact :", round(cosine(embed(q), embed(NEEDLES[2]["fact"])), 3))
print("cosine to a filler sentence :", round(cosine(embed(q), embed(FILLER[0])), 3))

## The answer oracle (no LLM, no leakage)

We deliberately do **not** call an LLM. Both strategies answer the same honest way: scan the text they are allowed to see and return the gold needle if it is present, else refuse. This isolates the variable we care about — *what text each strategy got to see* — from the messiness of a real generator, and removes any chance of leakage. Accuracy is then a plain substring match on the gold needle.

In [ ]:
def answer_from_text(question_idx, text):
    needle = NEEDLES[question_idx]["needle"]
    if needle.lower() in text.lower():
        return needle
    return "(not found in the provided text)"


def is_correct(question_idx, answer):
    return NEEDLES[question_idx]["needle"].lower() in answer.lower()


print(answer_from_text(0, "The headmaster of Starlight Academy is Professor Zephyrine Quorvax."))
print(answer_from_text(0, "The east tower houses the observatory."))

## Strategy A — the long-context LLM

Stuff **everything** into the window and scan it. On a clean needle it is perfectly accurate — but it pays for **every token in the corpus on every single query**. As the haystack grows, so does the bill (and the latency proxy = tokens it must scan).

In [ ]:
def llm_alone(question_idx, corpus):
    full_context = "\n".join(c["text"] for c in corpus)
    answer = answer_from_text(question_idx, full_context)
    tokens = sum(count_tokens(c["text"]) for c in corpus)
    return {
        "answer": answer,
        "correct": is_correct(question_idx, answer),
        "cost_tokens": tokens,      # billed: the whole corpus, every query
        "latency_proxy": tokens,    # proxy: tokens the model must scan
    }


r = llm_alone(2, small)
print("answer  :", r["answer"])
print("correct :", r["correct"])
print("cost    :", r["cost_tokens"], "tokens (the whole corpus)")

## Strategy B — RAG

Retrieve the top-k chunks by embedder cosine, then answer from **just those**. Cost is the retrieved tokens, which stays roughly flat as the corpus grows — you only ever read `k` chunks. The risk is the inverse of long-context's: if retrieval misses the needle, no amount of context can save you.

In [ ]:
def rag(question_idx, corpus, k=3):
    q_vec = embed(NEEDLES[question_idx]["question"])
    scored = sorted(((cosine(q_vec, embed(c["text"])), c) for c in corpus),
                    key=lambda s: -s[0])
    retrieved = [c for _score, c in scored[:k]]
    context = "\n".join(c["text"] for c in retrieved)
    answer = answer_from_text(question_idx, context)
    tokens = sum(count_tokens(c["text"]) for c in retrieved)
    return {
        "answer": answer,
        "correct": is_correct(question_idx, answer),
        "cost_tokens": tokens,      # billed: only the k retrieved chunks
        "latency_proxy": tokens,
        "retrieved_ids": [c["id"] for c in retrieved],
    }


r = rag(2, small, k=3)
print("retrieved:", r["retrieved_ids"])
print("answer   :", r["answer"], " correct:", r["correct"])
print("cost     :", r["cost_tokens"], "tokens (only k chunks)")

## The head-to-head table

Run every question under both strategies, across two corpus sizes, and tabulate **accuracy / cost / latency**. Watch what moves and what stays put as the corpus grows from a dozen chunks to forty.

In [ ]:
def run_head_to_head(filler_sizes, k=3):
    line = "=" * 78
    for n_filler in filler_sizes:
        corpus = build_corpus(n_filler)
        corpus_tokens = sum(count_tokens(c["text"]) for c in corpus)
        print(line)
        print(f"CORPUS: {len(corpus)} chunks  (~{corpus_tokens} tokens)   "
              f"[{len(NEEDLES)} needles, {n_filler} filler chunks per needle]")
        print(line)
        results = {"llm_alone": [], "rag": []}
        for qi in range(len(NEEDLES)):
            results["llm_alone"].append(llm_alone(qi, corpus))
            results["rag"].append(rag(qi, corpus, k=k))
        header = f"  {'strategy':<12}  {'accuracy':>8}  {'avg cost (tok)':>15}  {'avg latency (tok scanned)':>26}"
        print(header)
        print("  " + "-" * (len(header) - 2))
        for name in ("llm_alone", "rag"):
            rs = results[name]
            acc = sum(r["correct"] for r in rs) / len(rs)
            avg_cost = sum(r["cost_tokens"] for r in rs) / len(rs)
            avg_lat = sum(r["latency_proxy"] for r in rs) / len(rs)
            label = "LLM-alone" if name == "llm_alone" else f"RAG (k={k})"
            print(f"  {label:<12}  {acc:>7.0%}  {avg_cost:>15.1f}  {avg_lat:>26.1f}")
        print()


run_head_to_head(filler_sizes=[3, 12], k=3)

## Reading the table — and the honest takeaway

- **Accuracy is equal** here because the needle is clean and retrieval finds it. On a clean needle, *both* strategies are correct.
- **Cost and latency tell the real story.** LLM-alone pays for every token of the corpus on every query, so its bill climbs as the corpus grows (here ~125 → ~418 tokens). RAG reads only the `k` retrieved chunks, so its cost stays roughly flat (~32 → ~33).
- This is a **toy**. Real long-context models hit "lost in the middle" accuracy dips this substring oracle does not model; real retrieval can miss a *paraphrased* needle. The point is the **shape** of the tradeoff, not the exact numbers.

So the answer is **not** "RAG is dead" and **not** "RAG always wins" — it is **it depends**, on corpus size, query type, and budget. The pragmatic synthesis is to **route** between the two:

- **Self-Route** (Li et al., EMNLP 2024 industry track, arXiv 2407.16833) — let a cheap router send easy queries to long-context and hard ones to RAG.
- **LaRA** (Li et al., ICML 2025, arXiv 2502.09977) — a benchmark whose title says it plainly: *No Silver Bullet for LC or RAG Routing*.
- **U-NIAH** (arXiv 2503.00353) — the fictional-needle methodology this experiment borrows, so the measurement is leakage-free.

Back to the main Part 11 lesson: you don't settle this debate with vibes. You build a leakage-free eval set, measure accuracy *and* cost *and* latency, and let the numbers — not the hype — pick the architecture.